## Imports

In [33]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model

import xarray as xr

import numpy as np


## Load Data

In [34]:
is_local_data = False
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3

In [35]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [36]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")

approx_maximas = raw_mrsol_empirical_maximas.copy(deep=True)
approx_maximas["mrsol"] = (raw_mrsol_empirical_maximas["mrsol"] * 1.1).clip(min=1e-5)


In [37]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=4)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=4)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_test = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")


### functions to bring the data in shape

In [38]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = ds/maximas
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [39]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [40]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [41]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_var).sum() etc.
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [42]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [43]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [44]:
LinReg_mean = model.stats._parallel_linear_regression.ParLinearRegression()

In [45]:
LinReg_mean.fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

In [46]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)#Hier können noch sehr grosse werte auftauchen, wenn in irgendwelchen schichten die Maximas der verschieden runs sehr unterschiedlich sind.

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [47]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [48]:
residuals = LinReg_mean.residuals(var_predictors, var_target)

### Linear Regression of the Variance

In [49]:
LinReg_variance = model.stats._parallel_linear_regression.ParLinearRegression()

In [50]:
LinReg_variance.fit(predictors=var_predictors, target=(residuals.residuals)**2,location_dim="gridcell", regr_dim="time")

### Compute skewness samples

In [51]:
skew_target_noneT = shape_target(raw_mrsol_for_skew, approx_maximas)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Logit_Transform_ds(skew_target_noneT)


In [52]:
skew_predictors = shape_input(raw_input_for_skew,chunk_mask)

In [53]:
mean_prediction = LinReg_mean.predict(skew_predictors)

In [54]:
residuals = skew_target.mrsol - mean_prediction.prediction

In [55]:
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))

In [56]:
standardized_values_for_skew = (residuals/sigmas)

### Linear Regression of the Skewness

In [57]:
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()

In [58]:
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Export Prameters

In [59]:
if safe:
    model.save.save_params(LinReg_mean.params,LinReg_variance.params,skew_parameter=LinReg_skewness.params,maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/Transformation_Distribution/logit_transform/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
